# 04. CatBoost Baseline

This notebook establishes a reproducible CatBoost baseline on the minimally prepared dataset, evaluates threshold-dependent and probability-based metrics, and persists experiment artifacts.

In [1]:
from pathlib import Path
import sys

import pandas as pd

## Environment Setup

In [2]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root configured.")

PROCESSED_DIR = PROJECT_ROOT / "data/processed"
print("Processed directory configured.")

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
EXPERIMENTS_DIR = ARTIFACTS_DIR / "experiments"

EXPERIMENT_ID = "catboost_v0_raw_minimal_cv5"
RUN_TRAINING = False

Project root configured.
Processed directory configured.


In [3]:
from src.churn_ml.features import load_dataset
from src.churn_ml.modeling import (
    ExperimentConfig,
    load_experiment,
    save_experiment_result,
    summarize_target,
    build_experiment_summary,
)
from src.churn_ml.models.catboost import (
    CatBoostConfig,
    run_catboost_experiment,
)
from src.churn_ml.metrics import (
    evaluate_cross_fitted_thresholds,
    optimize_threshold_by_folds,
    summarize_threshold_plateau,
)

c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset Loading

In [4]:
dataset = load_dataset(
    version="v0_raw_minimal",
    processed_dir=PROCESSED_DIR,
)

print(f"Dataset version: {dataset.version}")
print(f"Train shape:     {dataset.X_train.shape}")
print(f"Target shape:    {dataset.y_train.shape}")
print(f"Test shape:      {dataset.X_test.shape}")

Dataset version: v0_raw_minimal
Train shape:     (10000, 205)
Target shape:    (10000,)
Test shape:      (2500, 205)


## Target Distribution

In [5]:
target_summary = summarize_target(dataset.y_train)

display(target_summary.style.format({"proportion": "{:.2%}"}))

,class,count,proportion
0,0,8695,86.95%
1,1,1305,13.05%


## CatBoost Baseline

In [6]:
catboost_config = CatBoostConfig.default()
experiment_config = ExperimentConfig.default()

if RUN_TRAINING:
    catboost_experiment = run_catboost_experiment(
        dataset=dataset,
        config=experiment_config,
        model_config=catboost_config,
        experiment_id=EXPERIMENT_ID,
    )

    save_experiment_result(
        result=catboost_experiment.result,
        experiments_dir=EXPERIMENTS_DIR,
        fold_metrics=catboost_experiment.fold_metrics,
        oof_predictions=catboost_experiment.oof_predictions,
        test_predictions=catboost_experiment.test_predictions,
        overwrite=True,
    )
else:
    catboost_experiment = load_experiment(
        experiment_id=EXPERIMENT_ID,
        experiments_dir=EXPERIMENTS_DIR,
    )

## Experiment Results

In [7]:
display(catboost_experiment.fold_metrics)

display(
    pd.Series(
        catboost_experiment.result.metrics,
        name="value",
    ).to_frame()
)

,fold,balanced_accuracy,roc_auc,average_precision,log_loss,decision_threshold,training_time_seconds,best_iteration
0,1,0.790897,0.970582,0.853891,0.149565,0.5,51.847443,609
1,2,0.770780,0.957099,0.809683,0.166769,0.5,157.620010,818
2,3,0.798078,0.960454,0.819719,0.164079,0.5,41.163835,473
3,4,0.822983,0.964896,0.841484,0.153321,0.5,274.152740,758
4,5,0.787159,0.958559,0.808492,0.166829,0.5,57.969307,611


,value
balanced_accuracy,0.793979
roc_auc,0.962040
average_precision,0.824716
log_loss,0.160113
decision_threshold,0.500000
optimized_threshold_oof,0.174000
optimized_balanced_accuracy_oof,0.894566
balanced_accuracy_mean,0.793979
balanced_accuracy_std,0.019054
roc_auc_mean,0.962318


In [8]:
summary = build_experiment_summary(catboost_experiment.result)

display(summary.style.format({"value": "{:.4f}"}))

,metric,value
0,Balanced Accuracy @ default threshold,0.7940
1,Optimized OOF Balanced Accuracy,0.8946
2,Optimized OOF threshold,0.1740
3,ROC-AUC,0.9620
4,Average Precision,0.8247
5,Log Loss,0.1601
6,"Training time, minutes",9.7239


In [9]:
fold_columns = [
    "fold",
    "balanced_accuracy",
    "roc_auc",
    "average_precision",
    "log_loss",
    "best_iteration",
    "training_time_seconds",
]

display(
    catboost_experiment.fold_metrics[fold_columns].style.format(
        {
            "balanced_accuracy": "{:.4f}",
            "roc_auc": "{:.4f}",
            "average_precision": "{:.4f}",
            "log_loss": "{:.4f}",
            "training_time_seconds": "{:.1f}",
        }
    )
)

,fold,balanced_accuracy,roc_auc,average_precision,log_loss,best_iteration,training_time_seconds
0,1,0.7909,0.9706,0.8539,0.1496,609,51.8
1,2,0.7708,0.9571,0.8097,0.1668,818,157.6
2,3,0.7981,0.9605,0.8197,0.1641,473,41.2
3,4,0.8230,0.9649,0.8415,0.1533,758,274.2
4,5,0.7872,0.9586,0.8085,0.1668,611,58.0


## Threshold Stability

In [10]:
fold_thresholds = optimize_threshold_by_folds(
    fold_metrics=catboost_experiment.fold_metrics,
    oof_predictions=catboost_experiment.oof_predictions,
)

display(
    fold_thresholds.style.format(
        {
            "optimized_threshold": "{:.3f}",
            "optimized_balanced_accuracy": "{:.4f}",
        }
    )
)

print(f"Threshold mean: {fold_thresholds['optimized_threshold'].mean():.3f}")

print(f"Threshold std:  {fold_thresholds['optimized_threshold'].std(ddof=1):.3f}")

,fold,optimized_threshold,optimized_balanced_accuracy
0,1,0.128,0.9135
1,2,0.184,0.8814
2,3,0.192,0.9020
3,4,0.169,0.9016
4,5,0.117,0.8864


Threshold mean: 0.158
Threshold std:  0.034


## Cross-Fitted Threshold Evaluation

For each holdout fold, the decision threshold is optimized using the remaining folds and then evaluated on the unseen holdout fold. This avoids evaluating a threshold on the same labels used to select it.

In [11]:
cross_fitted_result = evaluate_cross_fitted_thresholds(
    catboost_experiment.oof_predictions
)

cross_fitted_fold_metrics = cross_fitted_result.fold_metrics

display(
    cross_fitted_fold_metrics.style.format(
        {
            "threshold_cross_fitted": "{:.3f}",
            "balanced_accuracy_cross_fitted": "{:.4f}",
            "threshold_fold_optimal": "{:.3f}",
            "balanced_accuracy_fold_optimal": "{:.4f}",
            "balanced_accuracy_regret": "{:.4f}",
        }
    )
)

,fold,threshold_cross_fitted,balanced_accuracy_cross_fitted,threshold_fold_optimal,balanced_accuracy_fold_optimal,balanced_accuracy_regret
0,1,0.182,0.9009,0.128,0.9135,0.0125
1,2,0.174,0.8793,0.184,0.8814,0.0021
2,3,0.174,0.9011,0.192,0.9020,0.0009
3,4,0.175,0.8987,0.169,0.9016,0.0029
4,5,0.174,0.8830,0.117,0.8864,0.0033


In [12]:
optimized_oof_balanced_accuracy = catboost_experiment.result.metrics[
    "optimized_balanced_accuracy_oof"
]

optimism_gap = optimized_oof_balanced_accuracy - cross_fitted_result.balanced_accuracy

print(
    f"Cross-fitted OOF Balanced Accuracy: {cross_fitted_result.balanced_accuracy:.4f}"
)
print(
    "Mean cross-fitted threshold:         "
    f"{cross_fitted_fold_metrics['threshold_cross_fitted'].mean():.3f}"
)
print(
    "Cross-fitted threshold std:          "
    f"{cross_fitted_fold_metrics['threshold_cross_fitted'].std(ddof=1):.3f}"
)
print(
    "Mean Balanced Accuracy regret:       "
    f"{cross_fitted_fold_metrics['balanced_accuracy_regret'].mean():.4f}"
)
print(f"OOF threshold-selection optimism:    {optimism_gap:.4f}")

Cross-fitted OOF Balanced Accuracy: 0.8926
Mean cross-fitted threshold:         0.176
Cross-fitted threshold std:          0.003
Mean Balanced Accuracy regret:       0.0043
OOF threshold-selection optimism:    0.0020


## Threshold Plateau

This section evaluates how wide the near-optimal threshold region is. A wide plateau indicates that model performance is not sensitive to small threshold changes.

In [13]:
threshold_plateau = summarize_threshold_plateau(
    y_true=catboost_experiment.oof_predictions["target"],
    probabilities=catboost_experiment.oof_predictions["probability"].to_numpy(),
    tolerance=0.001,
)

threshold_plateau_summary = pd.DataFrame(
    [
        {
            "best_threshold": threshold_plateau.best_threshold,
            "best_balanced_accuracy": (threshold_plateau.best_balanced_accuracy),
            "lower_threshold": threshold_plateau.lower_threshold,
            "upper_threshold": threshold_plateau.upper_threshold,
            "midpoint_threshold": (threshold_plateau.midpoint_threshold),
            "plateau_width": threshold_plateau.width,
            "tolerance": threshold_plateau.tolerance,
        }
    ]
)

display(
    threshold_plateau_summary.style.format(
        {
            "best_threshold": "{:.3f}",
            "best_balanced_accuracy": "{:.4f}",
            "lower_threshold": "{:.3f}",
            "upper_threshold": "{:.3f}",
            "midpoint_threshold": "{:.3f}",
            "plateau_width": "{:.3f}",
            "tolerance": "{:.4f}",
        }
    )
)

,best_threshold,best_balanced_accuracy,lower_threshold,upper_threshold,midpoint_threshold,plateau_width,tolerance
0,0.174,0.8946,0.168,0.176,0.172,0.008,0.0010


In [14]:
fold_plateau_records = []

for fold_number in sorted(catboost_experiment.oof_predictions["fold"].unique()):
    fold_data = catboost_experiment.oof_predictions.loc[
        catboost_experiment.oof_predictions["fold"] == fold_number
    ]

    fold_plateau = summarize_threshold_plateau(
        y_true=fold_data["target"],
        probabilities=fold_data["probability"].to_numpy(),
        tolerance=0.001,
    )

    fold_plateau_records.append(
        {
            "fold": int(fold_number),
            "best_threshold": fold_plateau.best_threshold,
            "lower_threshold": fold_plateau.lower_threshold,
            "upper_threshold": fold_plateau.upper_threshold,
            "midpoint_threshold": (fold_plateau.midpoint_threshold),
            "plateau_width": fold_plateau.width,
            "best_balanced_accuracy": (fold_plateau.best_balanced_accuracy),
        }
    )

fold_plateaus = pd.DataFrame(fold_plateau_records)

display(
    fold_plateaus.style.format(
        {
            "best_threshold": "{:.3f}",
            "lower_threshold": "{:.3f}",
            "upper_threshold": "{:.3f}",
            "midpoint_threshold": "{:.3f}",
            "plateau_width": "{:.3f}",
            "best_balanced_accuracy": "{:.4f}",
        }
    )
)

,fold,best_threshold,lower_threshold,upper_threshold,midpoint_threshold,plateau_width,best_balanced_accuracy
0,1,0.128,0.127,0.129,0.128,0.002,0.9135
1,2,0.184,0.095,0.184,0.140,0.089,0.8814
2,3,0.192,0.172,0.192,0.182,0.020,0.9020
3,4,0.169,0.168,0.184,0.176,0.016,0.9016
4,5,0.117,0.094,0.117,0.105,0.023,0.8864


In [15]:
selected_threshold = catboost_experiment.result.metrics["optimized_threshold_oof"]

selected_threshold_inside_plateau = (
    threshold_plateau.lower_threshold
    <= selected_threshold
    <= threshold_plateau.upper_threshold
)

print(f"Selected OOF threshold:       {selected_threshold:.3f}")
print(
    "Near-optimal plateau:        "
    f"[{threshold_plateau.lower_threshold:.3f}, "
    f"{threshold_plateau.upper_threshold:.3f}]"
)
print(f"Selected threshold in plateau: {selected_threshold_inside_plateau}")

Selected OOF threshold:       0.174
Near-optimal plateau:        [0.168, 0.176]
Selected threshold in plateau: True


## Baseline Conclusion

The CatBoost baseline produces strong probability estimates, but the default classification threshold of 0.5 is inappropriate for the imbalanced target.

Cross-fitted threshold evaluation confirms that the performance gain from threshold optimization is stable and is not primarily caused by threshold overfitting. The selected threshold of 0.174 lies inside the global near-optimal plateau and is retained for test predictions.

In [16]:
final_threshold_summary = pd.DataFrame(
    [
        {
            "metric": "Balanced Accuracy @ 0.5",
            "value": catboost_experiment.result.metrics["balanced_accuracy"],
        },
        {
            "metric": "Optimized OOF Balanced Accuracy",
            "value": catboost_experiment.result.metrics[
                "optimized_balanced_accuracy_oof"
            ],
        },
        {
            "metric": "Cross-fitted Balanced Accuracy",
            "value": cross_fitted_result.balanced_accuracy,
        },
        {
            "metric": "Selected threshold",
            "value": selected_threshold,
        },
        {
            "metric": "Cross-fitted threshold mean",
            "value": cross_fitted_fold_metrics["threshold_cross_fitted"].mean(),
        },
        {
            "metric": "Threshold plateau lower bound",
            "value": threshold_plateau.lower_threshold,
        },
        {
            "metric": "Threshold plateau upper bound",
            "value": threshold_plateau.upper_threshold,
        },
        {
            "metric": "ROC-AUC",
            "value": catboost_experiment.result.metrics["roc_auc"],
        },
        {
            "metric": "Average Precision",
            "value": catboost_experiment.result.metrics["average_precision"],
        },
    ]
)

display(final_threshold_summary.style.format({"value": "{:.4f}"}))

,metric,value
0,Balanced Accuracy @ 0.5,0.7940
1,Optimized OOF Balanced Accuracy,0.8946
2,Cross-fitted Balanced Accuracy,0.8926
3,Selected threshold,0.1740
4,Cross-fitted threshold mean,0.1758
5,Threshold plateau lower bound,0.1680
6,Threshold plateau upper bound,0.1760
7,ROC-AUC,0.9620
8,Average Precision,0.8247


In [17]:
expected_test_predictions = (
    catboost_experiment.test_predictions["probability"] >= selected_threshold
).astype(int)

assert expected_test_predictions.equals(
    catboost_experiment.test_predictions["prediction_optimized_oof"]
)

test_positive_rate = expected_test_predictions.mean()

print(f"Selected threshold:       {selected_threshold:.3f}")
print(f"Test positive rate:       {test_positive_rate:.2%}")
print(f"Predicted positive rows:  {expected_test_predictions.sum():,}")
print(f"Total test rows:          {len(expected_test_predictions):,}")

Selected threshold:       0.174
Test positive rate:       20.80%
Predicted positive rows:  520
Total test rows:          2,500


In [18]:
experiment_dir = EXPERIMENTS_DIR / EXPERIMENT_ID
experiment_dir.mkdir(parents=True, exist_ok=True)

cross_fitted_fold_metrics.to_csv(
    experiment_dir / "cross_fitted_threshold_metrics.csv",
    index=False,
)

fold_plateaus.to_csv(
    experiment_dir / "threshold_plateaus_by_fold.csv",
    index=False,
)

final_threshold_summary.to_csv(
    experiment_dir / "threshold_summary.csv",
    index=False,
)

print(f"Threshold diagnostics saved to: {experiment_dir}")

Threshold diagnostics saved to: c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\artifacts\experiments\catboost_v0_raw_minimal_cv5


### Interpretation

- The default threshold of 0.5 produces a Balanced Accuracy of approximately 0.794.
- Global OOF threshold optimization increases Balanced Accuracy to approximately 0.895.
- Cross-fitted evaluation produces approximately 0.893, indicating only a small threshold-selection optimism of about 0.002.
- Cross-fitted thresholds are highly stable and concentrate around 0.174–0.176.
- The selected threshold of 0.174 lies inside the global near-optimal interval of 0.168–0.176.
- The CatBoost baseline is therefore retained with a decision threshold of 0.174.